# YOLO26 body-silhouette segmentation: fine-tuning, extraction and evaluation

**Pipeline step** 3 of 10 (person segmentation) · **Sample notebook** · **Target runtime** Kaggle GPU (T4) or any CUDA box; CPU works in smoke mode.

---

## 0.1 Purpose

Step 3 of the made-to-measure pipeline turns a camera capture (front A-pose + side view) into a
**binary body silhouette**. Step 6 (`model_training_measurement.ipynb`) consumes those silhouettes
in BodyM's convention: single-channel `uint8`, values `{0, 255}`, one person, whole body in frame.

This notebook:

1. prepares a person-silhouette dataset in Ultralytics YOLO segmentation format (§2),
2. imports a COCO-pretrained **YOLO26-seg** model and records its zero-shot baseline (§3),
3. fine-tunes it as a one-class (`person`) segmenter with augmentation chosen for capture conditions (§4),
4. wraps inference in a **silhouette extractor** with capture-quality gating and mask post-processing (§5),
5. evaluates region, boundary and width-profile accuracy against the pretrained baseline (§6),
6. exports the weights, an ONNX graph and a model card (§7).

## 0.2 How the references shape the method

| Reference | What it reports | What this notebook takes from it |
|---|---|---|
| Montazerian & Fol Leymarie (2022), 3DBODY.TECH | Single smartphone camera; body isolated with a pretrained MobileNet-SSD (OpenCV DNN); preprocessing chain of grey-level mapping, small-region removal, smoothing, edge detection and thresholding to fight reflections, shadows and low contrast; ≈94 % of pixels correct vs. hand segmentation; front view in A-pose, side view free; tight clothing; error grows below 0.5 m and above 3 m; plain and cluttered backgrounds; < ±1 cm mean upper-body error vs. tape (55 participants). | Photometric augmentation (§4), optional CLAHE contrast mapping as an ablation (§5–6), small-region removal + smoothing post-processing (§5), **pixel accuracy vs. hand-labelled masks** as a directly comparable metric (§6), capture checklist (§5.3). |
| Montazerian (2024), PhD thesis, Goldsmiths | Pipeline: isolate body → calibrate → extract features → place markers → refine; measurements from ellipses fitted to horizontal cross-sections head to hips; ≈±1 cm on 78 participants. | Silhouette **width profiles** at horizontal bands are the quantity downstream measurement depends on, so §6 scores them, not only IoU. |
| Sekar (2026), arXiv:2601.01639 | Compares landmark geometry, pose regression and object-calibrated front+side silhouettes; frames accepted only when landmark visibility > 0.9; waist as a Ramanujan ellipse from front and side widths; loose clothing inflates width, multiple people break the pipeline, low light degrades segmentation. | **Quality gating** in the extractor (exactly one person, whole body visible, confidence floor) (§5), ellipse circumference proxy in the width-profile evaluation (§6), failure-mode gallery (§6.4). |
| IEEE Xplore 11251570 | — | Not retrieved: IEEE Xplore was unreachable from the authoring environment. No method here is attributed to it. Check it before citing this notebook. |
| IEEE Xplore 10959938 | — | Same as above. |

Model choice: YOLO26 is the current Ultralytics generation and ships `-seg` checkpoints (`n/s/m/l/x`)
pretrained on COCO, whose class 0 is `person`. Fine-tuning from COCO keeps the person prior and
specialises the head and mask prototypes on full-body, single-subject captures.

## 0.3 Run modes

* `SMOKE = True` builds a small **synthetic** dataset (drawn A-pose figures on textured
  backgrounds), trains `yolo26n-seg` for 10 epochs at 320 px and runs every cell. It proves the
  code path, not the model. Results from it mean nothing.
* `SMOKE = False` uses `DATA_SOURCE`:
  * `"coco_person"`: full-body, single-subject images filtered from COCO-2017 keypoint annotations (§2.2),
  * `"mask_folder"`: your own captures with hand-labelled masks (`images/` + `masks/`, matching stems).
    This is the source to use once step 2 captures exist, because it matches deployment.

## §1 Setup

**WHAT** Install Ultralytics if missing, print the environment, set seeds, and define every tunable in one `CFG` dict.

**WHY** A single config at the top keeps the run reproducible and makes the ablations in §6 a matter of flipping keys. Ultralytics ≥ 8.4 is required for YOLO26 checkpoints.

In [ ]:
import importlib, subprocess, sys

def _ensure(pkg, spec=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec or pkg])

_ensure("ultralytics", "ultralytics>=8.4.0")
_ensure("pycocotools")
for _m, _s in (("pandas", None), ("tqdm", None), ("matplotlib", None), ("yaml", "pyyaml")):
    _ensure(_m, _s)
_ensure("cv2", "opencv-python-headless")

In [ ]:
import json, os, platform, random, re, shutil, time, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from tqdm.auto import tqdm
from ultralytics import YOLO

print("python      ", platform.python_version())
print("ultralytics ", ultralytics.__version__)
print("torch       ", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("opencv      ", cv2.__version__)

In [ ]:
SMOKE = True                 # flip to False for a real run
DATA_SOURCE = "coco_person"  # "coco_person" | "mask_folder" (ignored when SMOKE)

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "work"
WORK.mkdir(parents=True, exist_ok=True)

CFG = {
    "seed": 42,
    "work": WORK,
    "dataset_dir": WORK / "silhouette_ds",
    # ---- data ----------------------------------------------------------------------
    "coco_root": None,            # local COCO-2017 root (e.g. an attached Kaggle dataset); None -> download
    "coco_max_train": 4000,       # images kept after filtering (train2017 -> train/val)
    "coco_max_test": 500,         # images kept from val2017 -> test
    "coco_min_area_frac": 0.08,   # main person must cover at least this fraction of the image
    "coco_min_keypoints": 12,     # of 17; plus head and both ankles visible (full body)
    "coco_other_max_rel": 0.25,   # any other person must be smaller than this x main person
    "mask_folder": Path("/kaggle/input/my-captures"),  # images/ + masks/ with matching stems
    "subject_regex": r"^([^_]+)_",  # subject id from file stem, for subject-level splits
    "split_frac": (0.8, 0.1, 0.1),
    "border_margin_px": 2,        # bbox closer than this to the edge = truncated body
    # ---- model / training ------------------------------------------------------------
    "weights": "yolo26s-seg.pt",
    "imgsz": 640,
    "epochs": 60,
    "batch": 16,
    "patience": 15,
    "workers": 4,
    "mask_ratio": 2,              # default 4; finer prototype masks help boundary accuracy
    # augmentation (see §4 for the reasoning behind each value)
    "hsv_h": 0.015, "hsv_s": 0.6, "hsv_v": 0.5,
    "degrees": 3.0, "translate": 0.1, "scale": 0.35, "shear": 0.0, "perspective": 0.0002,
    "fliplr": 0.5, "flipud": 0.0,
    "mosaic": 0.5, "close_mosaic": 15, "mixup": 0.0, "copy_paste": 0.3,
    # ---- inference / post-processing --------------------------------------------------
    "conf": 0.35,
    "iou": 0.6,
    "postprocess": True,
    "morph_kernel_px": 5,
    "max_hole_frac": 0.002,       # fill enclosed holes smaller than this x mask area only
    "clahe": False,               # Montazerian-style contrast mapping before inference
    # ---- evaluation --------------------------------------------------------------------
    "bf_tolerances_px": (1, 2, 4),
    # height fractions from the top of the silhouette; approximate landmarks for an adult in A-pose
    "width_bands": {"chest": 0.28, "waist": 0.38, "hip": 0.50, "thigh": 0.58},
}

if SMOKE:
    CFG.update(weights="yolo26n-seg.pt", imgsz=320, epochs=10, batch=8, workers=0, patience=0,
               mosaic=0.0, close_mosaic=0, copy_paste=0.0,
               conf=0.01)  # the new 1-class head starts from scratch; after a few epochs its scores are still low

def set_seeds(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seeds(CFG["seed"])
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("SMOKE =", SMOKE, "| source =", "synthetic" if SMOKE else DATA_SOURCE, "| device =", DEVICE)
print("work dir:", WORK)

## §2 Dataset preparation

All sources end in the same layout, which Ultralytics reads directly:

```
silhouette_ds/
  images/{train,val,test}/<stem>.jpg
  labels/{train,val,test}/<stem>.txt   # "0 x1 y1 x2 y2 ..." normalised polygon per person
  silhouettes/{train,val,test}/<stem>.png  # GT mask of the *main* subject, {0,255}
  data.yaml
```

The extra `silhouettes/` folder holds the ground-truth silhouette of the subject the extractor
should return. §6 scores against it with pixel metrics, the way Montazerian & Fol Leymarie (2022)
scored against hand segmentation, which `mask mAP` alone does not capture.

### §2.1 Shared helpers: mask ↔ polygon

In [ ]:
from ultralytics.data.converter import merge_multi_segment

SPLITS = ("train", "val", "test")

def reset_dataset_dir(root: Path) -> None:
    if root.exists():
        shutil.rmtree(root)
    for sub in ("images", "labels", "silhouettes"):
        for s in SPLITS:
            (root / sub / s).mkdir(parents=True, exist_ok=True)

def mask_to_polygon(mask: np.ndarray, min_area_px: int = 50) -> Optional[np.ndarray]:
    # One normalised polygon (N,2) for a binary mask. Disjoint parts and holes are joined into a
    # single outline with merge_multi_segment, the same routine Ultralytics' COCO converter uses,
    # so one person stays one instance.
    h, w = mask.shape
    cnts, _ = cv2.findContours((mask > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = [c.reshape(-1, 2).astype(np.float32) for c in cnts if cv2.contourArea(c) >= min_area_px]
    if not cnts:
        return None
    cnts.sort(key=cv2.contourArea, reverse=True)
    pts = cnts[0] if len(cnts) == 1 else np.concatenate(merge_multi_segment([c.reshape(-1) for c in cnts]), 0)
    return pts / np.array([w, h], dtype=np.float32)

def write_label(path: Path, polygons: List[np.ndarray]) -> None:
    lines = ["0 " + " ".join(f"{v:.6f}" for v in np.clip(p, 0, 1).reshape(-1)) for p in polygons if len(p) >= 3]
    path.write_text("\n".join(lines) + ("\n" if lines else ""))

def write_sample(root: Path, split: str, stem: str, img_bgr: np.ndarray,
                 instance_masks: List[np.ndarray], main_idx: int = 0) -> None:
    cv2.imwrite(str(root / "images" / split / f"{stem}.jpg"), img_bgr, [cv2.IMWRITE_JPEG_QUALITY, 95])
    polys = [p for p in (mask_to_polygon(m) for m in instance_masks) if p is not None]
    write_label(root / "labels" / split / f"{stem}.txt", polys)
    cv2.imwrite(str(root / "silhouettes" / split / f"{stem}.png"),
                (instance_masks[main_idx] > 0).astype(np.uint8) * 255)

def write_data_yaml(root: Path) -> Path:
    p = root / "data.yaml"
    p.write_text(yaml.safe_dump({"path": str(root), "train": "images/train", "val": "images/val",
                                 "test": "images/test", "names": {0: "person"}}, sort_keys=False))
    return p

def split_by_group(groups: List[str], fracs=(0.8, 0.1, 0.1), seed: int = 42) -> Dict[str, str]:
    # Assign whole groups (subjects) to one split, so the same body never appears in train and test.
    uniq = sorted(set(groups))
    random.Random(seed).shuffle(uniq)
    n = len(uniq); a = int(round(fracs[0] * n)); b = a + int(round(fracs[1] * n))
    return {g: ("train" if i < a else "val" if i < b else "test") for i, g in enumerate(uniq)}

### §2.2 Source A: COCO-2017 full-body persons

**WHAT** Use the COCO *keypoint* annotations to keep only images that look like a measurement
capture: one dominant, non-crowd person whose head and both ankles are visible, who covers at
least `coco_min_area_frac` of the frame, whose box does not touch the image border, and with no
other person larger than `coco_other_max_rel` × the main one. All non-crowd persons are still
labelled so that small bystanders are not taught as background.

**WHY** Sekar (2026) accepts a frame only when the required landmarks are visible and reports that
multiple people break the pipeline. Montazerian & Fol Leymarie (2022) require the whole body in
view. Filtering COCO to that regime aligns the training distribution with deployment captures
better than training on every COCO person.

**NOTE** COCO is CC BY 4.0 for annotations; image licences vary per image (Flickr). Keep that in
mind before redistributing a derived dataset.

In [ ]:
COCO_ANN_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"

def coco_annotations(cache: Path) -> Path:
    if CFG["coco_root"] and (Path(CFG["coco_root"]) / "annotations").exists():
        return Path(CFG["coco_root"]) / "annotations"
    out = cache / "annotations"
    if not (out / "person_keypoints_train2017.json").exists():
        cache.mkdir(parents=True, exist_ok=True)
        zpath = cache / "annotations_trainval2017.zip"
        if not zpath.exists():
            torch.hub.download_url_to_file(COCO_ANN_URL, str(zpath))
        with zipfile.ZipFile(zpath) as z:
            for name in ("person_keypoints_train2017.json", "person_keypoints_val2017.json"):
                z.extract(f"annotations/{name}", cache)
    return out

# COCO keypoint order: 0 nose, 1-2 eyes, 3-4 ears, ..., 15 left ankle, 16 right ankle
HEAD_KPS, ANKLE_KPS = (0, 1, 2, 3, 4), (15, 16)

def select_full_body(coco, img_id: int) -> Optional[Tuple[dict, List[dict]]]:
    info = coco.imgs[img_id]
    W, H = info["width"], info["height"]
    anns = coco.loadAnns(coco.getAnnIds(imgIds=img_id, catIds=[1], iscrowd=None))
    if not anns or any(a["iscrowd"] for a in anns):
        return None
    main = max(anns, key=lambda a: a["area"])
    kp = np.array(main["keypoints"]).reshape(17, 3)
    x, y, w, h = main["bbox"]; m = CFG["border_margin_px"]
    ok = (main["num_keypoints"] >= CFG["coco_min_keypoints"]
          and (kp[list(HEAD_KPS), 2] > 0).any() and (kp[list(ANKLE_KPS), 2] > 0).all()
          and main["area"] >= CFG["coco_min_area_frac"] * W * H
          and x > m and y > m and x + w < W - m and y + h < H - m
          and all(a["area"] < CFG["coco_other_max_rel"] * main["area"] for a in anns if a is not main))
    return (main, anns) if ok else None

def fetch_image(info: dict, split_dir: str) -> Optional[np.ndarray]:
    if CFG["coco_root"]:
        p = Path(CFG["coco_root"]) / split_dir / info["file_name"]
        if p.exists():
            return cv2.imread(str(p))
    import urllib.request
    try:
        with urllib.request.urlopen(info["coco_url"], timeout=30) as r:
            return cv2.imdecode(np.frombuffer(r.read(), np.uint8), cv2.IMREAD_COLOR)
    except Exception as e:  # network hiccup: skip the image, do not abort the build
        print("skip", info["file_name"], type(e).__name__)
        return None

def build_coco_person(root: Path) -> None:
    from pycocotools.coco import COCO
    ann_dir = coco_annotations(CFG["work"] / "coco_cache")
    rng = random.Random(CFG["seed"])
    plan = []  # (coco, img_id, target split, image folder)
    for src, cap in (("train2017", CFG["coco_max_train"]), ("val2017", CFG["coco_max_test"])):
        coco = COCO(str(ann_dir / f"person_keypoints_{src}.json"))
        keep = [i for i in coco.getImgIds(catIds=[1]) if select_full_body(coco, i)]
        rng.shuffle(keep); keep = keep[:cap]
        print(f"{src}: {len(keep)} full-body single-subject images kept")
        if src == "train2017":
            n_val = int(len(keep) * CFG["split_frac"][1] / sum(CFG["split_frac"][:2]))
            plan += [(coco, i, "val" if k < n_val else "train", src) for k, i in enumerate(keep)]
        else:  # official val2017 is never seen in training: it is our test set
            plan += [(coco, i, "test", src) for i in keep]

    def _one(job):
        coco, img_id, split, src = job
        info = coco.imgs[img_id]
        img = fetch_image(info, src)
        if img is None:
            return
        main, anns = select_full_body(coco, img_id)
        anns = [main] + [a for a in anns if a is not main]
        write_sample(root, split, Path(info["file_name"]).stem, img, [coco.annToMask(a) for a in anns], 0)

    with ThreadPoolExecutor(16) as ex:
        list(tqdm(ex.map(_one, plan), total=len(plan), desc="writing COCO subset"))

### §2.3 Source B: your own captures (`mask_folder`)

**WHAT** Read `images/<stem>.(jpg|png)` and `masks/<stem>.png` (binary, foreground > 127), take
the subject id from the stem with `subject_regex`, and split **by subject**.

**WHY** Hand-labelled masks of your own front/side captures are the ground truth Montazerian &
Fol Leymarie (2022) evaluated against, and they match deployment lighting, phones and clothing.
Splitting by subject mirrors the measurement notebook: a front and a side photo of the same
person must not straddle train and test.

In [ ]:
def build_mask_folder(root: Path) -> None:
    src = Path(CFG["mask_folder"])
    imgs = sorted(p for p in (src / "images").iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
    pairs = [(p, src / "masks" / f"{p.stem}.png") for p in imgs]
    missing = [p.name for p, m in pairs if not m.exists()]
    assert not missing, f"{len(missing)} images have no mask, e.g. {missing[:5]}"
    rx = re.compile(CFG["subject_regex"])
    groups = [(rx.match(p.stem).group(1) if rx.match(p.stem) else p.stem) for p, _ in pairs]
    assign = split_by_group(groups, CFG["split_frac"], CFG["seed"])
    for (p, m), g in tqdm(list(zip(pairs, groups)), desc="writing captures"):
        img, mask = cv2.imread(str(p)), cv2.imread(str(m), cv2.IMREAD_GRAYSCALE)
        assert img.shape[:2] == mask.shape[:2], f"size mismatch for {p.name}"
        write_sample(root, assign[g], p.stem, img, [(mask > 127).astype(np.uint8)])
    print(pd.Series([assign[g] for g in groups]).value_counts().to_string())

### §2.4 Smoke source: synthetic A-pose figures

**WHAT** Draw stick-and-ellipse figures in an A-pose (front) or profile (side) on random textured
backgrounds with a soft shadow and a global brightness change.

**WHY** Only to exercise every cell offline in minutes. It is not a proxy for real people.

In [ ]:
def _figure_mask(h: int, w: int, rng: np.random.Generator, side: bool) -> np.ndarray:
    m = np.zeros((h, w), np.uint8)
    H = rng.uniform(0.70, 0.9) * h; top = rng.uniform(0.03, 0.97 * h - H) if h - H > 0.06 * h else 0.03 * h
    cx = rng.uniform(0.35, 0.65) * w; u = H / 8.0  # head-units
    sh = 0.55 if side else 1.0                      # profile is narrower
    cv2.ellipse(m, (int(cx), int(top + 0.5 * u)), (int(0.38 * u), int(0.5 * u)), 0, 0, 360, 255, -1)
    cv2.ellipse(m, (int(cx), int(top + 2.6 * u)), (int(1.0 * u * sh), int(1.6 * u)), 0, 0, 360, 255, -1)
    cv2.ellipse(m, (int(cx), int(top + 4.0 * u)), (int(0.95 * u * sh), int(0.9 * u)), 0, 0, 360, 255, -1)
    t = max(2, int(0.35 * u))
    for s in ((-1, 1) if not side else (1,)):
        hip = (int(cx + s * 0.45 * u * sh), int(top + 4.3 * u))
        cv2.line(m, hip, (int(cx + s * 0.75 * u * sh), int(top + 7.9 * u)), 255, int(1.6 * t))
        sho = (int(cx + s * 0.9 * u * sh), int(top + 1.6 * u))
        cv2.line(m, sho, (int(cx + s * (2.0 if not side else 0.3) * u), int(top + 4.2 * u)), 255, t)
    return m

def build_synthetic(root: Path, n: int = 120, h: int = 480, w: int = 360) -> None:
    rng = np.random.default_rng(CFG["seed"])
    for i in range(n):
        side = bool(i % 2)
        bg = cv2.GaussianBlur(rng.integers(0, 255, (h // 8, w // 8, 3), dtype=np.uint8), (0, 0), 1.0)
        img = cv2.resize(bg, (w, h), interpolation=cv2.INTER_CUBIC).astype(np.float32)
        m = _figure_mask(h, w, rng, side)
        shadow = cv2.GaussianBlur(np.roll(m, int(0.04 * w), 1), (0, 0), 9) / 255.0
        img *= (1 - 0.35 * shadow)[..., None]
        color = rng.uniform(30, 220, 3)
        img[m > 0] = 0.25 * img[m > 0] + 0.75 * color
        img = np.clip(img * rng.uniform(0.5, 1.3) + rng.normal(0, 6, img.shape), 0, 255).astype(np.uint8)
        split = "train" if i < 0.7 * n else "val" if i < 0.85 * n else "test"
        write_sample(root, split, f"syn{i:04d}_{'side' if side else 'front'}", img, [m])

In [ ]:
DS = CFG["dataset_dir"]
reset_dataset_dir(DS)
if SMOKE:
    build_synthetic(DS)
elif DATA_SOURCE == "coco_person":
    build_coco_person(DS)
elif DATA_SOURCE == "mask_folder":
    build_mask_folder(DS)
else:
    raise ValueError(DATA_SOURCE)
DATA_YAML = write_data_yaml(DS)

counts = {s: len(list((DS / "images" / s).glob("*.jpg"))) for s in SPLITS}
print("images per split:", counts)
for s in SPLITS:
    n_lbl = len(list((DS / "labels" / s).glob("*.txt")))
    assert n_lbl == counts[s] > 0, f"{s}: {counts[s]} images vs {n_lbl} label files"
print(DATA_YAML.read_text())

### §2.5 Visual check

Overlay the YOLO polygon labels (green) on a few training images. A label that misses limbs or
includes background here will be learned.

In [ ]:
def read_polygons(lbl: Path, w: int, h: int) -> List[np.ndarray]:
    out = []
    for line in lbl.read_text().splitlines():
        v = np.array(line.split()[1:], np.float32).reshape(-1, 2) * [w, h]
        out.append(v.astype(np.int32))
    return out

samples = sorted((DS / "images" / "train").glob("*.jpg"))[:8]
fig, axes = plt.subplots(2, 4, figsize=(14, 8))
for ax, p in zip(axes.ravel(), samples):
    img = cv2.imread(str(p)); h, w = img.shape[:2]
    cv2.polylines(img, read_polygons(DS / "labels" / "train" / f"{p.stem}.txt", w, h), True, (0, 255, 0), 2)
    ax.imshow(img[..., ::-1]); ax.set_title(p.stem, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## §3 Model import and zero-shot baseline

**WHAT** Load the COCO-pretrained YOLO26-seg checkpoint, print its summary, and keep it as the
baseline. COCO class 0 is `person`, so the pretrained model already segments people; §6
measures how much fine-tuning improves on it.

**WHY** A fine-tuned model has to beat the off-the-shelf one on *our* metrics, or the training
time bought nothing. Montazerian & Fol Leymarie (2022) used an off-the-shelf detector with
hand-tuned post-processing; the pretrained YOLO26 + §5 post-processing is the modern equivalent.

In [ ]:
base_model = YOLO(CFG["weights"])
base_model.info()
print("task:", base_model.task, "| classes:", len(base_model.names), "| class 0 =", base_model.names[0])
assert base_model.task == "segment" and base_model.names[0] == "person"

## §4 Fine-tuning

**WHAT** Train a one-class segmenter from the pretrained weights. Each augmentation value is a
decision about capture conditions:

| Arg | Value | Reason |
|---|---|---|
| `hsv_v`, `hsv_s` | 0.5, 0.6 | Montazerian & Fol Leymarie (2022): reflections, strong shadows and low contrast caused errors; Sekar (2026): low light degrades segmentation. Wide brightness/saturation jitter covers that. |
| `degrees` | 3 | Captures are upright; large rotations would teach poses the pipeline rejects. |
| `scale`, `translate` | 0.35, 0.1 | Montazerian reports errors below 0.5 m and above 3 m; scale jitter covers the usable distance range. |
| `fliplr` / `flipud` | 0.5 / 0 | Bodies are left-right symmetric enough; never upside down. |
| `copy_paste` | 0.3 | Pastes people onto other backgrounds; Montazerian tested plain and cluttered backgrounds. |
| `mosaic`, `close_mosaic` | 0.5, 15 | Mosaic crops bodies at tile edges, which contradicts the full-body prior, so it is reduced and switched off for the last 15 epochs. |
| `mask_ratio` | 2 | Prototype masks at 1/2 instead of 1/4 resolution. The measurement notebook's boundary-sensitivity phase shows step 6 error grows with boundary error, so boundary detail is worth the extra memory. |

Ultralytics rewrites `last.pt` every epoch under `WORK/runs/`; a rerun of this cell resumes from `last.pt`
if a previous session died.

In [ ]:
RUN_DIR = CFG["work"] / "runs"
RUN_NAME = "yolo26_silhouette" + ("_smoke" if SMOKE else "")
last_ckpt = RUN_DIR / RUN_NAME / "weights" / "last.pt"

TRAIN_ARGS = dict(
    data=str(DATA_YAML), epochs=CFG["epochs"], imgsz=CFG["imgsz"], batch=CFG["batch"],
    device=DEVICE, workers=CFG["workers"], project=str(RUN_DIR), name=RUN_NAME, exist_ok=True,
    seed=CFG["seed"], deterministic=True, patience=CFG["patience"], save_period=-1,
    mask_ratio=CFG["mask_ratio"], overlap_mask=True, cos_lr=True,
    **{k: CFG[k] for k in ("hsv_h", "hsv_s", "hsv_v", "degrees", "translate", "scale", "shear",
                           "perspective", "fliplr", "flipud", "mosaic", "close_mosaic", "mixup", "copy_paste")},
    plots=True, verbose=False,
)

t0 = time.time()
if last_ckpt.exists() and not SMOKE:
    print("resuming from", last_ckpt)
    YOLO(str(last_ckpt)).train(resume=True)
else:
    YOLO(CFG["weights"]).train(**TRAIN_ARGS)
print(f"training wall time: {(time.time() - t0) / 60:.1f} min")

BEST = RUN_DIR / RUN_NAME / "weights" / "best.pt"
assert BEST.exists(), BEST
ft_model = YOLO(str(BEST))
assert ft_model.names == {0: "person"}, ft_model.names

In [ ]:
res_csv = RUN_DIR / RUN_NAME / "results.csv"
hist = pd.read_csv(res_csv); hist.columns = [c.strip() for c in hist.columns]
cols = [c for c in hist.columns if c.startswith(("train/seg_loss", "val/seg_loss", "metrics/mAP50(M)", "metrics/mAP50-95(M)"))]
fig, axes = plt.subplots(1, len(cols), figsize=(4 * len(cols), 3))
for ax, c in zip(np.atleast_1d(axes), cols):
    ax.plot(hist["epoch"], hist[c]); ax.set_title(c, fontsize=9); ax.set_xlabel("epoch")
plt.tight_layout(); plt.show()

## §5 Silhouette extraction

### §5.1 Extractor

**WHAT** `extract_silhouette(image)` returns the subject's binary silhouette plus a status:

1. optional CLAHE on the L channel (Montazerian-style grey-level mapping; an ablation in §6),
2. YOLO26 inference at full image resolution (`retina_masks=True`), persons only,
3. **gating** (Sekar 2026; Montazerian & Fol Leymarie 2022):
   `no_person` when nothing passes `conf`; `multiple_people` when a second person is at least
   `coco_other_max_rel` × the main one; `truncated` when the box touches the frame edge,
4. **post-processing** (Montazerian & Fol Leymarie 2022 "removal of small regions and smoothing"):
   keep the largest connected component, fill only *small* enclosed holes (so the real gaps
   between arm and torso in an A-pose survive), then a morphological open + close,
5. output in BodyM's convention: `uint8`, `{0, 255}`, same size as the input.

A gated frame still returns its mask, so a capture UI can show the user what went wrong.

In [ ]:
def apply_clahe(img_bgr: np.ndarray) -> np.ndarray:
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def postprocess_mask(mask: np.ndarray, k: int, max_hole_frac: float) -> np.ndarray:
    m = (mask > 0).astype(np.uint8)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    if n > 1:
        m = (lab == 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])).astype(np.uint8)
    inv = 1 - m
    n, lab, stats, _ = cv2.connectedComponentsWithStats(inv, connectivity=4)
    h, w = m.shape; limit = max_hole_frac * m.sum()
    for i in range(1, n):
        x, y, bw, bh, area = stats[i]
        touches = x == 0 or y == 0 or x + bw == w or y + bh == h
        if not touches and area <= limit:
            m[lab == i] = 1
    if k > 1:
        ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        m = cv2.morphologyEx(cv2.morphologyEx(m, cv2.MORPH_OPEN, ker), cv2.MORPH_CLOSE, ker)
    return m

def extract_silhouette(model: YOLO, img_bgr: np.ndarray, postprocess: Optional[bool] = None,
                       clahe: Optional[bool] = None, conf: Optional[float] = None) -> Dict:
    postprocess = CFG["postprocess"] if postprocess is None else postprocess
    clahe = CFG["clahe"] if clahe is None else clahe
    h, w = img_bgr.shape[:2]
    src = apply_clahe(img_bgr) if clahe else img_bgr
    r = model.predict(src, imgsz=CFG["imgsz"], conf=conf or CFG["conf"], iou=CFG["iou"], classes=[0],
                      retina_masks=True, device=DEVICE, verbose=False)[0]
    out = {"mask": np.zeros((h, w), np.uint8), "status": "no_person", "conf": 0.0, "n_persons": 0}
    if r.masks is None or len(r.masks) == 0:
        return out
    masks = r.masks.data.cpu().numpy() > 0.5
    if masks.shape[1:] != (h, w):  # safety net; retina_masks should already match
        masks = np.stack([cv2.resize(m.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST) for m in masks]) > 0
    areas = masks.reshape(len(masks), -1).sum(1)
    i = int(np.argmax(areas))
    m = masks[i].astype(np.uint8)
    if postprocess:
        m = postprocess_mask(m, CFG["morph_kernel_px"], CFG["max_hole_frac"])
    x1, y1, x2, y2 = r.boxes.xyxy[i].tolist(); mg = CFG["border_margin_px"]
    others = [a for j, a in enumerate(areas) if j != i and a >= CFG["coco_other_max_rel"] * areas[i]]
    status = ("multiple_people" if others else
              "truncated" if (x1 <= mg or y1 <= mg or x2 >= w - mg or y2 >= h - mg) else "ok")
    out.update(mask=m * 255, status=status, conf=float(r.boxes.conf[i]), n_persons=int(len(masks)))
    return out

def extract_pair(model: YOLO, front_bgr: np.ndarray, side_bgr: np.ndarray) -> Dict[str, Dict]:
    # Front (A-pose) + side capture, the input step 6 expects.
    return {"front": extract_silhouette(model, front_bgr), "side": extract_silhouette(model, side_bgr)}

### §5.2 Qualitative check on held-out images

Columns: input, ground truth, pretrained YOLO26, fine-tuned YOLO26 (both post-processed). The
title carries the gate status.

In [ ]:
test_imgs = sorted((DS / "images" / "test").glob("*.jpg"))
show = test_imgs[:: max(1, len(test_imgs) // 4)][:4]
fig, axes = plt.subplots(len(show), 4, figsize=(12, 3.2 * len(show)), squeeze=False)
for row, p in zip(axes, show):
    img = cv2.imread(str(p)); gt = cv2.imread(str(DS / "silhouettes" / "test" / f"{p.stem}.png"), 0)
    b, f = extract_silhouette(base_model, img), extract_silhouette(ft_model, img)
    for ax, im, t in zip(row, (img[..., ::-1], gt, b["mask"], f["mask"]),
                         ("input", "ground truth", f"pretrained [{b['status']}]", f"fine-tuned [{f['status']}]")):
        ax.imshow(im, cmap=None if im.ndim == 3 else "gray"); ax.set_title(t, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

### §5.3 Capture checklist for step 2 (from the references)

The model cannot repair a bad capture. These come straight from the cited studies and should be
shown to the user at capture time:

* one person in frame, whole body from head to feet visible (Sekar 2026; Montazerian & Fol Leymarie 2022),
* front view in an A-pose; side view standing naturally (Montazerian & Fol Leymarie 2022),
* camera 0.5–3 m from the subject; errors rose outside that range (Montazerian & Fol Leymarie 2022),
* diffuse light, avoid strong shadows; low light degrades segmentation (both),
* close-fitting clothing at waist, shoulders and neck; loose clothing inflates silhouette width (both).

The gate statuses from §5.1 enforce the first item automatically.

## §6 Evaluation

### §6.1 Metrics

**WHAT** For each test image, compare the returned silhouette with the ground-truth silhouette of
the main subject:

* **IoU**, **Dice**, **pixel accuracy** (directly comparable with the ≈94 % reported by Montazerian & Fol Leymarie 2022),
* **boundary F-score** at 1/2/4 px and **HD95** (95th-percentile symmetric boundary distance, px),
* **width-profile error** at chest/waist/hip/thigh bands: horizontal width of the body run through
  the silhouette's centre column, as a % of silhouette height. These are the widths an ellipse
  model (Montazerian 2024; Sekar 2026) turns into girths, so they are closer to what step 6 needs
  than IoU is,
* **gate rate**: fraction of frames returned with a status other than `ok`.

Plus Ultralytics' `mask mAP50` / `mAP50-95` on the test split for the fine-tuned model.

**WHY** IoU on a 400-pixel-tall person can be 0.95 while the waist is 4 % too wide, and that 4 %
is roughly what separates the 9 mm and 25 mm tolerance bands in the measurement notebook.

In [ ]:
def _boundary(m: np.ndarray) -> np.ndarray:
    m = (m > 0).astype(np.uint8)
    return m - cv2.erode(m, np.ones((3, 3), np.uint8))

def boundary_scores(pred: np.ndarray, gt: np.ndarray, tols=(1, 2, 4)) -> Dict[str, float]:
    bp, bg = _boundary(pred), _boundary(gt)
    if bp.sum() == 0 or bg.sum() == 0:
        return {**{f"bf@{t}px": 0.0 for t in tols}, "hd95_px": float("nan")}
    dt_g = cv2.distanceTransform(1 - bg, cv2.DIST_L2, cv2.DIST_MASK_PRECISE)  # distance to nearest GT boundary pixel
    dt_p = cv2.distanceTransform(1 - bp, cv2.DIST_L2, cv2.DIST_MASK_PRECISE)
    d_pg, d_gp = dt_g[bp > 0], dt_p[bg > 0]
    out = {}
    for t in tols:
        prec, rec = (d_pg <= t).mean(), (d_gp <= t).mean()
        out[f"bf@{t}px"] = float(2 * prec * rec / (prec + rec + 1e-9))
    out["hd95_px"] = float(np.percentile(np.concatenate([d_pg, d_gp]), 95))
    return out

def width_profile(mask: np.ndarray, bands: Dict[str, float]) -> Dict[str, float]:
    # Width (px) of the foreground run that contains the body's centre column at each band.
    m = mask > 0
    rows = np.where(m.any(1))[0]
    if len(rows) == 0:
        return {k: np.nan for k in bands} | {"height_px": 0.0}
    top, bot = rows[0], rows[-1]; H = bot - top + 1
    cx = int(np.median(np.where(m)[1]))
    out = {"height_px": float(H)}
    for name, f in bands.items():
        r = m[int(top + f * H)]
        if not r[cx]:
            out[name] = np.nan; continue
        left = cx - np.argmin(r[cx::-1]) + 1 if not r[: cx + 1].all() else 0
        right = cx + np.argmin(r[cx:]) if not r[cx:].all() else len(r)
        out[name] = float(right - left)
    return out

def ramanujan_perimeter(a: float, b: float) -> float:
    # Ellipse perimeter from semi-axes; Sekar (2026) uses it for waist from front and side widths.
    return float(np.pi * (3 * (a + b) - np.sqrt((3 * a + b) * (a + 3 * b))))

def mask_metrics(pred: np.ndarray, gt: np.ndarray) -> Dict[str, float]:
    p, g = pred > 0, gt > 0
    inter, union = (p & g).sum(), (p | g).sum()
    out = {"iou": inter / union if union else 1.0,
           "dice": 2 * inter / (p.sum() + g.sum()) if (p.sum() + g.sum()) else 1.0,
           "pixel_acc": float((p == g).mean())}
    out.update(boundary_scores(pred, gt, CFG["bf_tolerances_px"]))
    wp, wg = width_profile(pred, CFG["width_bands"]), width_profile(gt, CFG["width_bands"])
    for k in CFG["width_bands"]:
        out[f"w_err_{k}_%h"] = 100 * abs(wp[k] - wg[k]) / wg["height_px"] if wg["height_px"] else np.nan
    out["height_err_%"] = 100 * abs(wp["height_px"] - wg["height_px"]) / wg["height_px"] if wg["height_px"] else np.nan
    return out

# Sanity checks on the metric code itself: identical masks score perfectly; a 3 px dilation is caught.
_g = np.zeros((200, 120), np.uint8); cv2.ellipse(_g, (60, 100), (30, 80), 0, 0, 360, 255, -1)
_s = mask_metrics(_g, _g)
assert abs(_s["iou"] - 1) < 1e-9 and _s["hd95_px"] == 0 and _s["bf@1px"] > 0.999
_d = mask_metrics(cv2.dilate(_g, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))), _g)
assert 2.5 <= _d["hd95_px"] <= 4.5 and _d["bf@1px"] < 0.2, _d
print("metric self-test passed:", {k: round(v, 3) for k, v in _d.items()})

### §6.2 Ablation table

Configurations: pretrained vs fine-tuned, with and without post-processing, and fine-tuned with
CLAHE. Every configuration sees the same test images.

In [ ]:
CONFIGS = {
    "pretrained (raw)":        (base_model, False, False),
    "pretrained + post":       (base_model, True,  False),
    "fine-tuned (raw)":        (ft_model,   False, False),
    "fine-tuned + post":       (ft_model,   True,  False),
    "fine-tuned + post + CLAHE": (ft_model, True,  True),
}

rows = []
for p in tqdm(test_imgs, desc="evaluating"):
    img = cv2.imread(str(p)); gt = cv2.imread(str(DS / "silhouettes" / "test" / f"{p.stem}.png"), 0)
    for name, (mdl, post, cl) in CONFIGS.items():
        r = extract_silhouette(mdl, img, postprocess=post, clahe=cl)
        rows.append({"config": name, "image": p.stem, "status": r["status"], "conf": r["conf"],
                     **mask_metrics(r["mask"], gt)})
EVAL = pd.DataFrame(rows)

summary = EVAL.groupby("config", sort=False).agg(
    n=("image", "size"), iou=("iou", "mean"), dice=("dice", "mean"), pixel_acc=("pixel_acc", "mean"),
    **{f"bf@{t}px": (f"bf@{t}px", "mean") for t in CFG["bf_tolerances_px"]},
    hd95_px=("hd95_px", "median"),
    **{f"w_err_{k}_%h": (f"w_err_{k}_%h", "median") for k in CFG["width_bands"]},
    gate_rate=("status", lambda s: (s != "ok").mean()))
pd.set_option("display.width", 200)
print(summary.round(4).to_string())
summary.to_csv(CFG["work"] / "silhouette_eval_summary.csv")
EVAL.to_csv(CFG["work"] / "silhouette_eval_per_image.csv", index=False)

In [ ]:
seg = ft_model.val(data=str(DATA_YAML), split="test", imgsz=CFG["imgsz"], batch=CFG["batch"],
                   device=DEVICE, plots=False, verbose=False).seg
MAP = {"mask_mAP50": float(seg.map50), "mask_mAP50-95": float(seg.map)}
print("fine-tuned, test split:", MAP)

### §6.3 Width-profile and ellipse-girth sensitivity

**WHAT** Show the distribution of width errors per band for the best configuration, then turn a
typical width error into a girth error with the Ramanujan ellipse used by Sekar (2026).

**WHY** This links segmentation quality to the tolerance bands in the measurement notebook: a
waist ellipse with front width 30 cm and side width 22 cm changes girth by roughly π/2 × the
width error, so a 1 % (≈1.7 cm on a 170 cm person) front-width error is already ≈2.6 cm of girth.

In [ ]:
best_cfg = summary["iou"].idxmax()
sub = EVAL[EVAL["config"] == best_cfg]
wcols = [f"w_err_{k}_%h" for k in CFG["width_bands"]]
fig, ax = plt.subplots(figsize=(7, 3))
ax.boxplot([sub[c].dropna() for c in wcols], showfliers=False)
ax.set_xticks(range(1, len(wcols) + 1), list(CFG["width_bands"]))
ax.set_ylabel("|width error| (% of height)"); ax.set_title(f"width-profile error, {best_cfg}")
plt.tight_layout(); plt.show()

# Girth sensitivity of a nominal waist (front 30 cm, side 22 cm) to the median front-width error.
H_CM = 170.0
med_err_cm = float(np.nanmedian(sub["w_err_waist_%h"])) / 100 * H_CM
g0 = ramanujan_perimeter(30 / 2, 22 / 2)
g1 = ramanujan_perimeter((30 + med_err_cm) / 2, 22 / 2)
print(f"best config: {best_cfg}")
print(f"median waist width error ~ {med_err_cm:.2f} cm on a {H_CM:.0f} cm person "
      f"-> waist girth {g0:.1f} cm becomes {g1:.1f} cm (delta {g1 - g0:+.2f} cm)")

### §6.4 Failure gallery

The worst six fine-tuned predictions by IoU, with the gate status. Read these before trusting the
averages: the references point to loose clothing, shadows and extra people as the usual causes.

In [ ]:
worst = sub.nsmallest(6, "iou")
fig, axes = plt.subplots(2, len(worst), figsize=(2.6 * len(worst), 6), squeeze=False)
for j, r in enumerate(worst.itertuples()):
    img = cv2.imread(str(DS / "images" / "test" / f"{r.image}.jpg"))
    gt = cv2.imread(str(DS / "silhouettes" / "test" / f"{r.image}.png"), 0)
    pr = extract_silhouette(ft_model, img)["mask"]
    over = img[..., ::-1].copy()
    over[(pr > 0) & (gt == 0)] = (255, 60, 60)   # false positive: red
    over[(pr == 0) & (gt > 0)] = (60, 60, 255)   # false negative: blue
    axes[0][j].imshow(img[..., ::-1]); axes[0][j].set_title(f"{r.image}\n{r.status}", fontsize=8)
    axes[1][j].imshow(over); axes[1][j].set_title(f"IoU {r.iou:.3f}", fontsize=8)
    for a in axes[:, j]: a.axis("off")
plt.tight_layout(); plt.show()
print(EVAL[EVAL["config"] == best_cfg]["status"].value_counts().to_string())

## §7 Export

**WHAT** Copy `best.pt`, export an ONNX graph (YOLO26 exports end-to-end without a separate NMS
step), and write a model card with the config, the evaluation summary and the references.

**WHY** Step 3 in production runs on whatever device step 2 captures on; ONNX keeps that open.
The model card is the record of how these weights were produced.

In [ ]:
EXPORT = CFG["work"] / "export_silhouette"
EXPORT.mkdir(exist_ok=True)
shutil.copy2(BEST, EXPORT / "yolo26_silhouette_best.pt")
try:
    onnx_path = Path(YOLO(str(BEST)).export(format="onnx", imgsz=CFG["imgsz"], dynamic=False))
    shutil.copy2(onnx_path, EXPORT / onnx_path.name)
except Exception as e:  # onnx is optional; do not lose the run over it
    onnx_path = None
    print("ONNX export skipped:", type(e).__name__, e)

card = {
    "model": "YOLO26-seg fine-tuned for single-subject body silhouettes",
    "base_weights": CFG["weights"], "ultralytics": ultralytics.__version__,
    "smoke_run": SMOKE, "data_source": "synthetic" if SMOKE else DATA_SOURCE,
    "images_per_split": counts,
    "train_args": {k: (str(v) if isinstance(v, Path) else v) for k, v in TRAIN_ARGS.items()},
    "inference": {k: CFG[k] for k in ("imgsz", "conf", "iou", "postprocess", "morph_kernel_px", "max_hole_frac", "clahe")},
    "output_convention": "uint8 single channel, values {0,255}, input resolution (BodyM-compatible)",
    "test_mask_map": MAP,
    "test_summary": json.loads(summary.round(5).to_json(orient="index")),
    "best_config": best_cfg,
    "references": [
        "Montazerian, M., & Fol Leymarie, F. (2022). Measuring the human body from a single camera, with applications to the clothing and fashion industry. 3DBODY.TECH 2022. https://proc.3dbody.tech/abstracts/2022/2211montazerian.html",
        "Montazerian, M. (2024). Designing a contactless, AI system to measure the human body using a single camera for the clothing and fashion industry (Doctoral thesis). Goldsmiths, University of London. https://research.gold.ac.uk/id/eprint/35887/",
        "Sekar, G. (2026). An empirical study of monocular human body measurement under weak calibration. arXiv:2601.01639. https://arxiv.org/abs/2601.01639",
        "IEEE Xplore document 11251570 (not retrieved; https://ieeexplore.ieee.org/document/11251570)",
        "IEEE Xplore document 10959938 (not retrieved; https://ieeexplore.ieee.org/document/10959938)",
    ],
}
(EXPORT / "model_card.json").write_text(json.dumps(card, indent=2, default=str))
print(sorted(p.name for p in EXPORT.iterdir()))

## §8 Using the extractor downstream

```python
from ultralytics import YOLO
model = YOLO("export_silhouette/yolo26_silhouette_best.pt")
pair = extract_pair(model, cv2.imread("front.jpg"), cv2.imread("side.jpg"))
if all(v["status"] == "ok" for v in pair.values()):
    cv2.imwrite("front_mask.png", pair["front"]["mask"])   # -> step 4 / step 6
    cv2.imwrite("side_mask.png",  pair["side"]["mask"])
else:
    print({k: v["status"] for k, v in pair.items()})       # ask the user to retake
```

**Known limits**
* COCO people are not measurement captures; the `mask_folder` source with your own hand-labelled
  front/side images is what the final model should be trained and, above all, evaluated on.
* The width bands are fixed height fractions, not landmarks. They are good enough to compare
  segmenters, not to report body measurements.
* Two of the five requested references (the IEEE Xplore papers) could not be read when this
  notebook was written; nothing here relies on them.